# 06 — Interprétabilité : t-SNE et UMAP (Partie 6)
**Responsable : DIOP Alioune** · relecture : FALL Madjiguène

Mise au point sur le modèle V1 (MLP [64]) et la **validation** : aucun accès au test. Les fonctions de `src/har/interpret.py` sont ensuite réutilisées telles quelles dans la V2.

## 6.1 Lectures
- **t-SNE** : principe … ; limites (distances entre groupes non interprétables, perplexité, stochasticité) …
- **UMAP** : principe … ; différences avec t-SNE …

In [ ]:
import sys; sys.path.append("../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import torch
from torch import nn
from har import config, data, models, training, metrics, plots, interpret
training.set_seed(0)
print("device :", training.DEVICE)

In [ ]:
df = data.load_train()
df_tr, df_va, val_subjects = data.split_by_subject(df)
print("volontaires de validation :", val_subjects)
assert set(df_tr.subject) & set(df_va.subject) == set()
X_tr, y_tr, X_va, y_va, scaler = data.prepare(df_tr, df_va)
train_loader, val_loader = data.make_loaders(X_tr, y_tr, X_va, y_va, seed=0)
IN_DIM = X_tr.shape[1]
X_tr.shape, X_va.shape

## Modèle V1 entraîné

In [ ]:
training.set_seed(0)
model = models.MLP(IN_DIM, [64], activation="relu")   # TODO : config V1 retenue
opt = training.make_optimizer("adam", model.parameters(), lr=1e-3)
hist = training.fit(model, train_loader, val_loader, opt, epochs=50)
ev = training.evaluate(model, val_loader, nn.CrossEntropyLoss(), return_preds=True)
plots.confusion_heatmap(metrics.confusion(ev["y_true"], ev["y_pred"]), "V1 — validation", save="p6_confusion_val")

## 6.2 Projections : entrées brutes vs dernière couche cachée

In [ ]:
interpret.compare_spaces(model, X_va, y_va, method="tsne", perplexity=30, save="p6_tsne_v1")
interpret.compare_spaces(model, X_va, y_va, method="umap", save="p6_umap_v1")

## 6.3 Limites : perplexité et variabilité d'une exécution à l'autre

In [ ]:
H = interpret.hidden_activations(model, X_va)
interpret.perplexity_grid(H, y_va, save="p6_perplexity")
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, s in zip(ax, [0, 1, 2]):
    plots.scatter_2d(interpret.project(H, "tsne", seed=s, perplexity=30), y_va, f"graine {s}", a)

## Analyse
- Classes qui se chevauchent vs classes confondues dans la matrice : …
- Ce que le réseau a changé par rapport à l'espace d'entrée : …
- Limites et précautions : …